# Experiment 3: Data Transformation (Python)
**Course:** 26AI2003 – Statistical Programming for Clinical Data Analytics

## Aim
To merge related clinical datasets on a common patient identifier and to create derived variables from the merged dataset, in order to produce an analysis-ready dataset.

## Description
Clinical data is stored in separate tables. Here, `clinical_demographics.csv` (patient details recorded at enrollment) and `clinical_vitals_labs.csv` (blood pressure, heart rate, weight, BMI and lab values) are combined on the key `USUBJID`. The notebook then checks that the merge did not add or drop patients and creates four derived variables:

| Derived variable | Type | Logic |
|---|---|---|
| `AGE_GROUP` | recode of one column | AGE < 40 → "<40", 40–59 → "40-59", 60 or more → "60+" |
| `BP_CATEGORY` | from two columns | SYSBP ≥ 130 or DIABP ≥ 80 → "High"; else SYSBP ≥ 120 → "Elevated"; else "Normal" |
| `DAYS_ON_STUDY` | date difference | cutoff date (2025-12-31) minus `ENROLL_DATE`, in days |
| `HIGH_RISK_FLAG` | composite flag | "Yes" if BP_CATEGORY is "High" **and** (AGE ≥ 60 **or** BMI ≥ 30) |

**Needs:** `pandas`, `numpy`. Keep both CSV files in the same folder as this notebook.

## Step 1: Import libraries

In [ ]:
import pandas as pd
import numpy as np

## Step 2: Load both datasets

In [ ]:
demographics = pd.read_csv("clinical_demographics.csv")
vitals_labs = pd.read_csv("clinical_vitals_labs.csv")

print("Demographics:", demographics.shape)
print("Vitals/labs :", vitals_labs.shape)
demographics.head()

## Step 3: Check the merge key is unique in both datasets
If the number of unique IDs equals the number of rows, each patient appears once.

In [ ]:
print("Demographics: rows =", len(demographics), ", unique USUBJID =", demographics["USUBJID"].nunique())
print("Vitals/labs : rows =", len(vitals_labs),  ", unique USUBJID =", vitals_labs["USUBJID"].nunique())

## Step 4: Merge on USUBJID
An inner join keeps only patients found in both datasets.

In [ ]:
df = pd.merge(demographics, vitals_labs, on="USUBJID", how="inner")
df.head()

## Step 5: Verify the merge did not add or drop records

In [ ]:
print("Merged shape:", df.shape)
print("Same row count as demographics:", len(df) == len(demographics))
print("Same row count as vitals/labs :", len(df) == len(vitals_labs))

# Which patients (if any) would be lost by the inner join?
only_demo = set(demographics["USUBJID"]) - set(vitals_labs["USUBJID"])
only_vit  = set(vitals_labs["USUBJID"]) - set(demographics["USUBJID"])
print("In demographics only:", len(only_demo))
print("In vitals/labs only :", len(only_vit))

## Step 6: Convert ENROLL_DATE to a date
Needed so dates can be subtracted.

In [ ]:
df["ENROLL_DATE"] = pd.to_datetime(df["ENROLL_DATE"])
df["ENROLL_DATE"].dtype

## Step 7: Derived variable by recoding – AGE_GROUP

In [ ]:
def age_group(age):
    if age < 40:
        return "<40"
    elif age < 60:
        return "40-59"
    else:
        return "60+"

df["AGE_GROUP"] = df["AGE"].apply(age_group)
df[["AGE", "AGE_GROUP"]].head()

## Step 8: Derived variable from two columns – BP_CATEGORY

In [ ]:
def bp_category(row):
    if row["SYSBP"] >= 130 or row["DIABP"] >= 80:
        return "High"
    elif row["SYSBP"] >= 120:
        return "Elevated"
    else:
        return "Normal"

df["BP_CATEGORY"] = df.apply(bp_category, axis=1)
df[["SYSBP", "DIABP", "BP_CATEGORY"]].head()

## Step 9: Date-based derived variable – DAYS_ON_STUDY

In [ ]:
cutoff_date = pd.Timestamp("2025-12-31")

df["DAYS_ON_STUDY"] = (cutoff_date - df["ENROLL_DATE"]).dt.days
df[["ENROLL_DATE", "DAYS_ON_STUDY"]].head()

## Step 10: Composite flag – HIGH_RISK_FLAG
"Yes" when blood pressure is High **and** the patient is aged 60 or more **or** has a BMI of 30 or more.

In [ ]:
condition = (df["BP_CATEGORY"] == "High") & ((df["AGE"] >= 60) | (df["BMI"] >= 30))

df["HIGH_RISK_FLAG"] = np.where(condition, "Yes", "No")
df[["AGE", "BMI", "BP_CATEGORY", "HIGH_RISK_FLAG"]].head()

## Step 11: Review the derived variables

In [ ]:
print("AGE_GROUP:");      print(df["AGE_GROUP"].value_counts(), "\n")
print("BP_CATEGORY:");    print(df["BP_CATEGORY"].value_counts(), "\n")
print("HIGH_RISK_FLAG:"); print(df["HIGH_RISK_FLAG"].value_counts(), "\n")
print("DAYS_ON_STUDY:");  print(df["DAYS_ON_STUDY"].describe().round(1))

## Step 12: Save the analysis dataset

In [ ]:
df.to_csv("clinical_trial_data_analysis.csv", index=False)
print("Saved: clinical_trial_data_analysis.csv", df.shape)